## 0. Cài đặt & cấu hình chung

In [ ]:
# Chạy cell này đầu tiên
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# Thư mục lưu hình
OUT_DIR = "report_figures"
os.makedirs(OUT_DIR, exist_ok=True)

# Cấu hình hiển thị chung cho báo cáo
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# Bảng màu nhất quán
SENT_ORDER  = ["POSITIVE", "NEGATIVE", "NEUTRAL", "CONFLICT"]
SENT_COLORS = {
    "POSITIVE": "#2e7d32",
    "NEGATIVE": "#c62828",
    "NEUTRAL":  "#f9a825",
    "CONFLICT": "#6a1b9a",
    "NONE":     "#9e9e9e",
}
ASPECTS = ["HOTEL", "LOCATION", "ROOMS", "FACILITIES", "FOOD&DRINKS", "SERVICE"]
MODEL_COLORS = {"PhoBERT": "#1f77b4", "ViSoBERT": "#ff7f0e"}

def save_fig(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path)
    print("✅ Đã lưu:", path)
    return path

# Đường dẫn dữ liệu trên Drive (chỉ cần khi muốn vẽ Hình 3.7 / 4.5 hoặc tính lại EDA từ CSV)
# Chỉnh lại nếu thư mục DACS của bạn nằm chỗ khác
DACS_DIR     = "/content/drive/MyDrive/DACS"
LABEL_READY  = os.path.join(DACS_DIR, "absa_label_ready.csv")
SPLIT_DIR    = os.path.join(DACS_DIR, "absa_splits")
PHOBERT_EVAL = os.path.join(DACS_DIR, "phobert_absa_final_loss_balanced", "test_evaluation")

def try_read_csv(path):
    # Doc CSV neu ton tai, nguoc lai tra None (an toan khi chua mount Drive)
    if os.path.exists(path):
        try:
            return pd.read_csv(path)
        except Exception as e:
            print("⚠️ Lỗi đọc", path, ":", e)
    return None

print("✅ Sẵn sàng. Hình sẽ được lưu vào thư mục:", os.path.abspath(OUT_DIR))

### (Tuỳ chọn) Mount Google Drive


In [ ]:
# Tuỳ chọn — bỏ qua nếu không cần đọc CSV
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Đã mount Drive. Kiểm tra nhanh các file:")
    for p in [LABEL_READY, SPLIT_DIR, PHOBERT_EVAL]:
        print(("  ✓ " if os.path.exists(p) else "  ✗ thiếu: ") + p)
except Exception as e:
    print("⚠️ Không mount được Drive (có thể đang chạy ngoài Colab):", e)

---
# A. Phân tích dữ liệu (Chương 3)


## Hình 3.2 — Phân bố số aspect trên mỗi unit
Minh hoạ cho mục **3.5.1**: phần lớn unit chỉ đề cập 1 aspect.

In [ ]:
# Số aspect "active" (khác NONE) trên mỗi unit
df_lab = try_read_csv(LABEL_READY)
if df_lab is not None and all(a in df_lab.columns for a in ASPECTS):
    active = (df_lab[ASPECTS] != "NONE").sum(axis=1)
    dist = active.value_counts().sort_index()
    counts = {int(k): int(v) for k, v in dist.items()}
    print("Tính lại từ CSV.")
else:
    # Số liệu in sẵn trong báo cáo (mục 3.5.1)
    counts = {1: 6411, 2: 777, 3: 282, 4: 99, 5: 25, 6: 1}
    print("Dùng số liệu in sẵn trong báo cáo.")

xs = list(counts.keys())
ys = list(counts.values())
total = sum(ys)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(xs, ys, color="#1f77b4", edgecolor="white")
for b, y in zip(bars, ys):
    ax.text(b.get_x() + b.get_width()/2, y + total*0.01,
            f"{y:,}\n({y/total*100:.1f}%)", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("Số aspect được gán trên mỗi unit")
ax.set_ylabel("Số lượng unit")
ax.set_title("Phân bố số aspect trên mỗi đơn vị văn bản")
ax.set_xticks(xs)
ax.set_ylim(0, max(ys)*1.15)
ax.grid(axis="x", visible=False)
plt.tight_layout()
save_fig(fig, "hinh_3_2_phan_bo_so_aspect.png")
plt.show()

## Hình 3.3 — Phân bố sentiment tổng thể
Minh hoạ cho mục **3.5.2**: chỉ đếm các aspect thực sự xuất hiện (bỏ NONE).

In [ ]:
from collections import Counter
df_lab = try_read_csv(LABEL_READY)
if df_lab is not None and all(a in df_lab.columns for a in ASPECTS):
    c = Counter()
    for a in ASPECTS:
        c.update(df_lab.loc[df_lab[a] != "NONE", a].tolist())
    sent_counts = {k: int(c.get(k, 0)) for k in SENT_ORDER}
    print("Tính lại từ CSV.")
else:
    sent_counts = {"POSITIVE": 5233, "NEGATIVE": 2913, "NEUTRAL": 555, "CONFLICT": 637}
    print("Dùng số liệu in sẵn trong báo cáo.")

labels = SENT_ORDER
vals = [sent_counts[k] for k in labels]
total = sum(vals)
cols = [SENT_COLORS[k] for k in labels]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, vals, color=cols, edgecolor="white")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + total*0.01,
            f"{v:,}\n({v/total*100:.1f}%)", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Số lượng nhãn")
ax.set_title("Phân bố sentiment tổng thể (chỉ tính aspect xuất hiện)")
ax.set_ylim(0, max(vals)*1.15)
ax.grid(axis="x", visible=False)
plt.tight_layout()
save_fig(fig, "hinh_3_3_phan_bo_sentiment_tong.png")
plt.show()

## Hình 3.4 — Heatmap sentiment theo aspect
Minh hoạ cho mục **3.5.3** (số liệu bảng phân bố sentiment theo từng aspect). Bỏ cột NONE để thấy rõ tương quan giữa 4 nhãn cảm xúc thật.

In [ ]:
# Ma trận aspect x sentiment (số liệu mục 3.5.3)
report_table = {
    "HOTEL":       {"POSITIVE": 3326, "NEGATIVE": 1157, "NEUTRAL": 223, "CONFLICT": 109},
    "LOCATION":    {"POSITIVE": 500,  "NEGATIVE": 68,   "NEUTRAL": 10,  "CONFLICT": 23},
    "ROOMS":       {"POSITIVE": 360,  "NEGATIVE": 697,  "NEUTRAL": 107, "CONFLICT": 369},
    "FACILITIES":  {"POSITIVE": 197,  "NEGATIVE": 277,  "NEUTRAL": 63,  "CONFLICT": 25},
    "FOOD&DRINKS": {"POSITIVE": 221,  "NEGATIVE": 215,  "NEUTRAL": 77,  "CONFLICT": 74},
    "SERVICE":     {"POSITIVE": 629,  "NEGATIVE": 499,  "NEUTRAL": 75,  "CONFLICT": 37},
}

df_lab = try_read_csv(LABEL_READY)
if df_lab is not None and all(a in df_lab.columns for a in ASPECTS):
    mat = np.array([[int((df_lab[a] == s).sum()) for s in SENT_ORDER] for a in ASPECTS])
    print("Tính lại từ CSV.")
else:
    mat = np.array([[report_table[a][s] for s in SENT_ORDER] for a in ASPECTS])
    print("Dùng số liệu in sẵn trong báo cáo.")

fig, ax = plt.subplots(figsize=(8, 5.5))
im = ax.imshow(mat, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(SENT_ORDER)), labels=SENT_ORDER)
ax.set_yticks(range(len(ASPECTS)), labels=ASPECTS)
# Annotate số trong từng ô
thresh = mat.max() * 0.55
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i, j]:,}", ha="center", va="center",
                color="white" if mat[i, j] > thresh else "black", fontsize=10)
ax.set_title("Heatmap số lượng sentiment theo từng aspect")
ax.grid(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Số lượng nhãn")
plt.tight_layout()
save_fig(fig, "hinh_3_4_heatmap_sentiment_aspect.png")
plt.show()

## Hình 3.5 (bonus) — Tỷ lệ sentiment theo aspect (100% stacked)
Minh hoạ trực tiếp cho mục **3.5.4 — Nhận xét về mất cân bằng dữ liệu**. Thể hiện *cơ cấu* sentiment trong mỗi aspect, làm nổi bật ROOMS nhiều CONFLICT/NEGATIVE còn LOCATION thiên POSITIVE.

In [ ]:
# Dùng lại ma trận mat (aspect x [POS,NEG,NEU,CONFLICT]) từ cell Hình 3.4
row_sum = mat.sum(axis=1, keepdims=True)
prop = mat / np.where(row_sum == 0, 1, row_sum) * 100  # phần trăm theo hàng

fig, ax = plt.subplots(figsize=(10, 5.5))
left = np.zeros(len(ASPECTS))
y = np.arange(len(ASPECTS))
for j, s in enumerate(SENT_ORDER):
    vals = prop[:, j]
    ax.barh(y, vals, left=left, color=SENT_COLORS[s], label=s, edgecolor="white")
    for i, v in enumerate(vals):
        if v >= 6:
            ax.text(left[i] + v/2, y[i], f"{v:.0f}%", ha="center", va="center",
                    color="white", fontsize=9)
    left += vals
ax.set_yticks(y, labels=ASPECTS)
ax.invert_yaxis()
ax.set_xlabel("Tỷ lệ trong các nhãn cảm xúc của aspect (%)")
ax.set_title("Cơ cấu sentiment theo từng aspect (mất cân bằng dữ liệu)")
ax.set_xlim(0, 100)
ax.grid(axis="y", visible=False)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
plt.tight_layout()
save_fig(fig, "hinh_3_5_co_cau_sentiment_theo_aspect.png")
plt.show()

## Hình 3.6 — So sánh độ dài token giữa hai tokenizer
Minh hoạ cho mục **3.8** (bảng thống kê độ dài token). Biểu đồ cột nhóm theo các phân vị, kèm đường ngưỡng `MAX_LENGTH` đã chọn cho mỗi mô hình.

In [ ]:
# Số liệu phân vị độ dài token (mục 3.8.1 và 3.8.2)
metrics = ["Trung bình", "Trung vị", "P90", "P95", "P99", "Lớn nhất"]
phobert  = [12.96, 11, 23, 29, 47, 128]
visobert = [26.04, 21, 48, 61.30, 97.06, 214]
PHO_MAX, VISO_MAX = 64, 128

x = np.arange(len(metrics))
w = 0.38
fig, ax = plt.subplots(figsize=(10, 5.5))
b1 = ax.bar(x - w/2, phobert,  w, label="PhoBERT",  color=MODEL_COLORS["PhoBERT"])
b2 = ax.bar(x + w/2, visobert, w, label="ViSoBERT", color=MODEL_COLORS["ViSoBERT"])
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 3,
                f"{b.get_height():.0f}", ha="center", va="bottom", fontsize=8)
ax.axhline(PHO_MAX,  color=MODEL_COLORS["PhoBERT"],  ls="--", lw=1.2, alpha=0.8)
ax.axhline(VISO_MAX, color=MODEL_COLORS["ViSoBERT"], ls="--", lw=1.2, alpha=0.8)
ax.text(-0.4, PHO_MAX+4,  "MAX_LENGTH PhoBERT = 64",  color=MODEL_COLORS["PhoBERT"],  fontsize=8, ha="left")
ax.text(-0.4, VISO_MAX+4, "MAX_LENGTH ViSoBERT = 128", color=MODEL_COLORS["ViSoBERT"], fontsize=8, ha="left")
ax.set_xticks(x, labels=metrics)
ax.set_ylabel("Độ dài (số token)")
ax.set_title("So sánh độ dài token: PhoBERT vs ViSoBERT")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False)
plt.tight_layout()
save_fig(fig, "hinh_3_6_so_sanh_do_dai_token.png")
plt.show()

## Hình 3.7 (bonus) — Phân bố nhãn giữa train / validation / test
Minh hoạ cho mục **3.7.4 — Kiểm tra chất lượng chia tập**: so sánh *tỷ lệ* sentiment ở 3 tập để xác nhận chia tập cân bằng.
👉 Cần mount Drive và có sẵn `absa_splits/{train,validation,test}.csv`. Nếu không có file, cell tự bỏ qua.

In [ ]:
ID_TO_LABEL = {1: "POSITIVE", 2: "NEGATIVE", 3: "NEUTRAL", 4: "CONFLICT"}
splits = {
    "train":      try_read_csv(os.path.join(SPLIT_DIR, "train.csv")),
    "validation": try_read_csv(os.path.join(SPLIT_DIR, "validation.csv")),
    "test":       try_read_csv(os.path.join(SPLIT_DIR, "test.csv")),
}

if any(v is None for v in splits.values()):
    print("⚠️ Không tìm thấy đủ file split CSV -> bỏ qua Hình 3.7.")
    print("   Mount Drive và kiểm tra lại đường dẫn SPLIT_DIR nếu muốn vẽ hình này.")
else:
    id_cols = [f"{a}_id" for a in ASPECTS]
    rows = []
    for name, d in splits.items():
        n = len(d)
        for lid, lname in ID_TO_LABEL.items():
            cnt = int(sum((d[c].astype(int) == lid).sum() for c in id_cols))
            rows.append({"split": name, "sentiment": lname, "ratio": cnt / n})
    rdf = pd.DataFrame(rows)
    pivot = rdf.pivot(index="sentiment", columns="split", values="ratio").reindex(SENT_ORDER)
    pivot = pivot[["train", "validation", "test"]]

    x = np.arange(len(SENT_ORDER)); w = 0.26
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for i, sp in enumerate(["train", "validation", "test"]):
        ax.bar(x + (i-1)*w, pivot[sp].values, w, label=sp.capitalize())
    ax.set_xticks(x, labels=SENT_ORDER)
    ax.set_ylabel("Tỷ lệ nhãn / số unit của tập")
    ax.set_title("Phân bố sentiment giữa train / validation / test")
    ax.grid(axis="x", visible=False)
    ax.legend(frameon=False)
    plt.tight_layout()
    save_fig(fig, "hinh_3_7_phan_bo_nhan_theo_tap.png")
    plt.show()

---
# B. Kết quả thực nghiệm (Chương 4) — bổ sung
ABSA.ipynb đã có biểu đồ so sánh metric tổng (4.1), Macro F1 theo aspect (4.2) và confusion matrix (4.3). Hai hình dưới đây bổ sung thêm.

## Hình 4.4 (bonus) — Chênh lệch Macro F1: PhoBERT − ViSoBERT theo aspect
Bổ sung cho mục **4.4**: biểu đồ phân kỳ cho thấy ngay PhoBERT thắng ở 5/6 aspect, ViSoBERT chỉ nhỉnh hơn ở LOCATION.

In [ ]:
# Macro F1 theo aspect (mục 4.4.1)
pho  = {"HOTEL":0.7285,"LOCATION":0.5120,"ROOMS":0.7051,"FACILITIES":0.5157,"FOOD&DRINKS":0.7575,"SERVICE":0.6914}
viso = {"HOTEL":0.6585,"LOCATION":0.5339,"ROOMS":0.6380,"FACILITIES":0.4096,"FOOD&DRINKS":0.6632,"SERVICE":0.6431}

diff = {a: pho[a] - viso[a] for a in ASPECTS}
order = sorted(ASPECTS, key=lambda a: diff[a])
vals = [diff[a] for a in order]
colors = ["#2e7d32" if v >= 0 else "#c62828" for v in vals]

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(order, vals, color=colors, edgecolor="white")
for b, v in zip(bars, vals):
    ax.text(v + (0.004 if v >= 0 else -0.004), b.get_y() + b.get_height()/2,
            f"{v:+.3f}", va="center", ha="left" if v >= 0 else "right", fontsize=9)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Macro F1 PhoBERT − ViSoBERT")
ax.set_title("Chênh lệch Macro F1 theo aspect (xanh: PhoBERT tốt hơn)")
ax.grid(axis="y", visible=False)
lim = max(abs(min(vals)), abs(max(vals))) * 1.3
ax.set_xlim(-lim, lim)
plt.tight_layout()
save_fig(fig, "hinh_4_4_chenh_lech_macro_f1.png")
plt.show()

## Hình 4.5 (bonus) — Heatmap Classification Report của PhoBERT
Minh hoạ cho mục **4.5**: gom Precision / Recall / F1 của từng lớp ở cả 6 aspect vào một heatmap, dễ nhìn ra lớp nào (NEUTRAL, CONFLICT…) yếu.
👉 Đọc các file `classification_report_{aspect}.csv` do ABSA.ipynb đã lưu. Tự bỏ qua nếu không có.

In [ ]:
metric = "f1-score"   # đổi thành "precision" hoặc "recall" nếu muốn
reports = {}
for a in ASPECTS:
    fname = f"classification_report_{a}.csv".replace("&", "&")
    d = try_read_csv(os.path.join(PHOBERT_EVAL, fname))
    if d is not None:
        reports[a] = d

if not reports:
    print("⚠️ Không tìm thấy file classification_report_*.csv -> bỏ qua Hình 4.5.")
    print("   Kiểm tra lại PHOBERT_EVAL:", PHOBERT_EVAL)
else:
    classes = ["NONE", "POSITIVE", "NEGATIVE", "NEUTRAL", "CONFLICT"]
    grid = np.full((len(ASPECTS), len(classes)), np.nan)
    for i, a in enumerate(ASPECTS):
        d = reports.get(a)
        if d is None:
            continue
        idx_col = d.columns[0]
        d = d.set_index(idx_col)
        for j, c in enumerate(classes):
            if c in d.index and metric in d.columns:
                grid[i, j] = float(d.loc[c, metric])

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    im = ax.imshow(grid, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(classes)), labels=classes)
    ax.set_yticks(range(len(ASPECTS)), labels=ASPECTS)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if not np.isnan(grid[i, j]):
                ax.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center", fontsize=9,
                        color="black")
    ax.set_title(f"Hình 4.5 — {metric} của PhoBERT theo lớp × aspect")
    ax.grid(False)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label(metric)
    plt.tight_layout()
    save_fig(fig, "hinh_4_5_classification_report_phobert.png")
    plt.show()

---
# C. Sơ đồ minh hoạ (Chương 2 & 3)
Hai sơ đồ này không lấy từ dữ liệu mà mô tả thiết kế hệ thống. Bạn có thể chèn thẳng vào báo cáo hoặc vẽ lại đẹp hơn bằng draw.io.

## Hình 2.3 — Kiến trúc mô hình multi-aspect classifier
Theo mô tả mục **3.9**: Encoder (PhoBERT/ViSoBERT) → Dropout → Linear (30) → reshape `[B, 6, 5]` → 6 đầu ra sentiment.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off")
ax.set_xlim(0, 12); ax.set_ylim(0, 6)

def box(x, y, w, h, text, fc, tc="white", fs=10):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04,rounding_size=0.12",
                 fc=fc, ec="black", lw=1.2))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", color=tc, fontsize=fs, weight="bold")

def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>",
                 mutation_scale=15, lw=1.4, color="#444"))

# Luồng chính
box(0.2, 2.4, 1.6, 1.2, "Văn bản\n(text_model_input)", "#607d8b", fs=8.5)
box(2.1, 2.4, 1.6, 1.2, "Tokenizer\n(PhoBERT/\nViSoBERT)", "#455a64", fs=8.5)
box(4.0, 2.1, 1.7, 1.8, "Encoder\npretrained\n(Transformer)", "#1f77b4", fs=9.5)
box(6.0, 2.85, 1.3, 0.95, "Dropout", "#8e24aa", fs=9.5)
box(6.0, 1.6,  1.3, 0.95, "Linear\n6×5 = 30", "#d84315", fs=8.5)
box(7.6, 1.6, 1.8, 2.2, "Reshape\n[B, 6, 5]", "#2e7d32", fs=10)

arrow(1.8, 3.0, 2.1, 3.0)
arrow(3.7, 3.0, 4.0, 3.0)
arrow(5.7, 3.0, 6.0, 3.3)
arrow(6.65, 2.85, 6.65, 2.55)
arrow(7.3, 2.1, 7.6, 2.1)

# 6 đầu ra aspect (mỗi cái 1 softmax 5 lớp)
ys = np.linspace(4.35, 0.55, len(ASPECTS))
for a, yy in zip(ASPECTS, ys):
    box(10.0, yy-0.28, 1.85, 0.56, f"{a}\n(5 lớp)", "#00897b", fs=7.5)
    arrow(9.4, 2.7, 10.0, yy)

ax.text(10.92, 5.05, "6 đầu ra sentiment", ha="center", fontsize=9, style="italic", color="#00695c")
ax.set_title("Kiến trúc multi-output classifier cho ABSA", fontsize=13, weight="bold")
plt.tight_layout()
save_fig(fig, "hinh_2_3_kien_truc_mo_hinh.png")
plt.show()

## Hình 3.1 — Sơ đồ quy trình xử lý dữ liệu
Tóm tắt pipeline mục **3.x**: từ dữ liệu thô tới dataset sẵn sàng huấn luyện.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.axis("off")
ax.set_xlim(0, 12); ax.set_ylim(0, 3)

steps = [
    ("Dữ liệu thô\n8.793 unit", "#607d8b"),
    ("Làm sạch +\nlọc trùng\n→ 7.595", "#00897b"),
    ("Chuẩn hoá nhãn\n6 aspect × 5 lớp", "#3949ab"),
    ("Tách từ\nVnCoreNLP", "#8e24aa"),
    ("Chia tập\ntrain/val/test", "#d84315"),
    ("Tokenize\n→ dataset", "#2e7d32"),
]
w, h, gap = 1.7, 1.4, 0.28
x = 0.15
for i, (txt, fc) in enumerate(steps):
    ax.add_patch(FancyBboxPatch((x, 0.8), w, h, boxstyle="round,pad=0.04,rounding_size=0.12",
                 fc=fc, ec="black", lw=1.2))
    ax.text(x + w/2, 0.8 + h/2, txt, ha="center", va="center", color="white", fontsize=9, weight="bold")
    if i < len(steps) - 1:
        ax.add_patch(FancyArrowPatch((x + w, 1.5), (x + w + gap, 1.5),
                     arrowstyle="-|>", mutation_scale=16, lw=1.6, color="#444"))
    x += w + gap

ax.set_title("Quy trình tiền xử lý & chuẩn bị dữ liệu", fontsize=13, weight="bold")
plt.tight_layout()
save_fig(fig, "hinh_3_1_quy_trinh_xu_ly_du_lieu.png")
plt.show()

---
# 💾 Tải tất cả hình về máy
Nén toàn bộ thư mục `report_figures/` thành 1 file zip để tải về và chèn vào báo cáo Word.

In [ ]:
import shutil
zip_path = shutil.make_archive("report_figures", "zip", OUT_DIR)
print("✅ Đã nén:", zip_path)
print("\nDanh sách hình đã tạo:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  -", f)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("\n(Tải thủ công nếu không ở Colab):", zip_path)